In [ ]:
# =========================================================
#  1. Setup
# =========================================================
!nvidia-smi

!pip install -U peft==0.10.0
!pip install -U accelerate==0.30.1
!pip install -U transformers datasets bitsandbytes
!pip install -q trl

import os
os.makedirs("logs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("✅ Setup complete!")

In [ ]:
# =========================================================
#  2. Model & Tokenizer Loading（Mistral7Bロード）
# =========================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

In [ ]:
# =========================================================
#  3. Dataset Loading
# =========================================================
from datasets import load_dataset
import json

# training_data.json はアップロード済みを想定
dataset = load_dataset("json", data_files="training_data.json")

print("✅ Dataset loaded:", dataset)

In [ ]:
# =========================================================
#  4. Preprocessing
# =========================================================
def preprocess_function(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    text_pairs = [f"User: {inp}\nBot: {out}" for inp, out in zip(inputs, outputs)]
    
    model_inputs = tokenizer(
        text_pairs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )
    
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

processed_dataset = dataset["train"].map(preprocess_function, batched=True)
print("✅ Tokenization complete!")

In [ ]:
# =========================================================
#  5. LoRA Configuration
# =========================================================
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA configuration applied.")

In [ ]:
# =========================================================
#  6. Training
# =========================================================
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    tokenizer=tokenizer
)

trainer.train()
print("✅ Training finished.")

In [ ]:
# =========================================================
#  7. Save Model
# =========================================================
OUTPUT_DIR = "mistral7b_finetuned"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

In [ ]:
# =========================================================
#  8. Inference Test
# =========================================================
from transformers import pipeline

OUTPUT_DIR = "outputs/mistral-lora"

pipe = pipeline(
    "text-generation",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device_map="auto"
)

prompt = "User: What are the latest trends in artificial intelligence?\nBot:"
result = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)

print("💬 Generated Response:")
print(result[0]["generated_text"])


In [ ]:
# =========================================================
#  9. Logging & Export
# =========================================================

with open("logs/training_summary.txt", "w") as f:
    f.write("Training complete.\n")
    f.write(f"Model saved to: {OUTPUT_DIR}\n")
    f.write("Last prompt:\n" + prompt + "\n\nResponse:\n" + response)

print("✅ Logs saved to logs/training_summary.txt")